In [1]:
## import
import pandas as pd
import sqlite3
import os

In [2]:
##Load your prepared datasets and Check columns

airbnb_df = pd.read_csv("../../Data_Wrangling_Datasets/christchurch_airbnb_with_sa2.csv")
bond_df= pd.read_csv("../../Data_Wrangling_Datasets/cleaned_bond_data_aligned.csv")

print(bond_df.shape)
print(airbnb_df.shape)

print("Bond columns:")
print(bond_df.columns.tolist())

print("\nAirbnb columns:")
print(airbnb_df.columns.tolist())

(26991, 12)
(28758, 21)
Bond columns:
['TimeFrame', 'Location Id', 'Dwelling Type', 'Number Of Beds', 'Total Bonds', 'Active Bonds', 'Closed Bonds', 'Median Rent', 'Geometric Mean Rent', 'Upper Quartile Rent', 'Lower Quartile Rent', 'Log Std Dev Weekly Rent']

Airbnb columns:
['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm', 'month_year', 'TimeFrame', 'area_name', 'sa2_code']


In [10]:
# Clean and standardize SA2 Code / Location Id strings
def clean_id(val):
    if pd.isna(val):
        return ""
    # Convert floats like '316700.0' or numbers to clean integer strings
    s = str(val).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

airbnb_df['sa2_code_clean'] = airbnb_df['sa2_code'].apply(clean_id)
bond_df['Location_Id_clean'] = bond_df['Location Id'].apply(clean_id)

# Convert TimeFrame columns to datetime objects for accurate date matching
airbnb_df['TimeFrame_dt'] = pd.to_datetime(airbnb_df['TimeFrame'], dayfirst=True, errors='coerce')
bond_df['TimeFrame_dt'] = pd.to_datetime(bond_df['TimeFrame'], dayfirst=True, errors='coerce')


In [11]:
## Create SQLite database

conn = sqlite3.connect("../../Data_Wrangling_Datasets/rental_data.sqlite")

In [12]:
## Store datasets in SQLite

bond_df.to_sql("bond_data", conn, if_exists="replace", index=False)
airbnb_df.to_sql("airbnb_data", conn, if_exists="replace", index=False)

28758

In [13]:
## Verify SQLite tables
pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

,name
0,bond_data
1,airbnb_data


In [15]:
## SQL Join
query = """
SELECT *
FROM bond_data AS b
INNER JOIN airbnb_data AS a
    ON b.TimeFrame_dt = a.TimeFrame_dt AND b."Location_Id_clean" = a.sa2_code_clean
"""

joined_df = pd.read_sql_query(query, conn)
joined_df.head()

,TimeFrame,Location Id,Dwelling Type,Number Of Beds,Total Bonds,Active Bonds,Closed Bonds,Median Rent,Geometric Mean Rent,Upper Quartile Rent,...,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,month_year,TimeFrame,area_name,sa2_code,sa2_code_clean,TimeFrame_dt
0,2026-01-01,316800,House,1,6,3,0,290,281,380,...,1.23,1,337,15,Jan-26,01-01-2026,Clearwater,316800,316800,2026-01-01 00:00:00
1,2026-01-01,316800,House,1,6,3,0,290,281,380,...,1.24,1,344,14,Feb-26,01-01-2026,Clearwater,316800,316800,2026-01-01 00:00:00
2,2026-01-01,316800,House,1,6,3,0,290,281,380,...,1.29,1,353,17,Mar-26,01-01-2026,Clearwater,316800,316800,2026-01-01 00:00:00
3,2026-01-01,316800,House,1,6,3,0,290,281,380,...,0.38,1,298,1,Jan-26,01-01-2026,Clearwater,316800,316800,2026-01-01 00:00:00
4,2026-01-01,316800,House,1,6,3,0,290,281,380,...,0.38,1,321,1,Feb-26,01-01-2026,Clearwater,316800,316800,2026-01-01 00:00:00


In [16]:
# Inspect the result and export

print("Shape:", joined_df.shape)
joined_df.info()

print("Shape:", joined_df.shape)
joined_df.info()

Shape: (60895, 37)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60895 entries, 0 to 60894
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   TimeFrame                       60895 non-null  object 
 1   Location Id                     60895 non-null  int64  
 2   Dwelling Type                   60895 non-null  object 
 3   Number Of Beds                  60895 non-null  object 
 4   Total Bonds                     60895 non-null  int64  
 5   Active Bonds                    60895 non-null  int64  
 6   Closed Bonds                    60895 non-null  int64  
 7   Median Rent                     60895 non-null  int64  
 8   Geometric Mean Rent             60895 non-null  int64  
 9   Upper Quartile Rent             60895 non-null  int64  
 10  Lower Quartile Rent             60895 non-null  int64  
 11  Log Std Dev Weekly Rent         60895 non-null  float64
 12  Location_Id_c

In [17]:
## Close DB connection

conn.close()